## Patient Recall Churn — Data Profiling & Validation


### Set Up

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np


In [ ]:
# project root = one level up from notebooks/
ROOT = Path.cwd().parent
RAW = ROOT / "data_raw"
WORKING = ROOT / "data_working"

In [ ]:
print("root:", ROOT)
print("raw exists:", RAW.exists())
for f in sorted(RAW.iterdir()):
    print(repr(f.name), "<-- dir" if f.is_dir() else "")

### Source tables

In [ ]:
tables = sorted(f for f in RAW.glob("*.xlsx") if not f.name.startswith("~$"))

for f in tables:
    xl = pd.ExcelFile(f)
    print(f"\n=== {f.name} ===")
    print("sheets:", xl.sheet_names)
    df = pd.read_excel(f, sheet_name=xl.sheet_names[0])
    print("shape:", df.shape)
    print("columns:", df.columns.tolist())

### Join-key validation

In [ ]:
#checking if pseudonymise worked

pat = pd.read_csv(WORKING / "PatientExport.csv")
app = pd.read_csv(WORKING / "AppointmentExport.csv")

pat_keys = set(pat["patient_key"])
app_keys = set(app["patient_key"])
overlap  = pat_keys & app_keys      # keys present in BOTH tables

print("patients:", len(pat_keys))
print("distinct patients in appointments:", len(app_keys))
print("overlap (appt keys that match a patient):", len(overlap))
print("share of appointment patients matched:",
      round(len(overlap) / len(app_keys), 3))

### Distributions and Coverage

In [ ]:
#loading the four working tables

pat = pd.read_csv(WORKING / "PatientExport.csv")
app = pd.read_csv(WORKING / "AppointmentExport.csv")
rx  = pd.read_csv(WORKING / "PatientRX.csv")
orders = pd.read_csv(WORKING / "Order_Report.csv")

In [ ]:
# low memory = read the whole column at once, then decide the type

rx = pd.read_csv(WORKING / "PatientRX.csv", low_memory=False)

In [ ]:
#how is attendence encoded

# grab the column -> tally the distinct values -> print the counts

print(app["Status"].value_counts(dropna=False))

In [ ]:
#recall-anchor coverage

print("RX rows:", len(rx))
print("ReaMonths populated:", rx["ReaMonths"].notna().sum())  #how many months until the patient's next recall is due
print("ReDate populated:", rx["ReDate"].notna().sum())  #how many dates until the patient's next recall is due
print(rx["ReaMonths"].value_counts(dropna=False).head(10))  #top 10 most common recall anchor values

In [ ]:
#date ranges 

#changing the date strings to datetime objects
#then printing the min and max of each column, along with the number of unparseable values


for df, col, name in [(app,"AppointDate","appointments"),
                      (orders,"OrderDate","orders"),
                      (rx,"ReDate","recall due")]:
    d = pd.to_datetime(df[col], errors="coerce")
    print(f"{name:12} {d.min()} -> {d.max()}  ({d.isna().sum()} unparseable)")

In [ ]:
#which recall anchors are actually labellable? (i.e. have a real date, and the window has closed before the end of the data)


redate = pd.to_datetime(rx["ReDate"], errors="coerce")

data_end = pd.Timestamp("2026-06-19")   # last appointment in your data
grace = pd.DateOffset(months=6)         # your chosen grace window

# sentinel nulls
is_sentinel = redate <= pd.Timestamp("1901-01-01")    #True/False column flagging fake value (VisionPlus placeholder for "no recall due")
print("sentinel/placeholder ReDates:", is_sentinel.sum())

# future recalls whose window hasn't closed yet -> unlabellable
window_closes = redate + grace
still_pending = window_closes > data_end
print("recall window not yet closed (unlabellable):", (still_pending & ~is_sentinel).sum())

# what's actually labellable: real date, window closed before data ends
labellable = (~is_sentinel) & (~still_pending)
print("labellable RX records:", labellable.sum())
print("\nvalid ReDate range:",
      redate[~is_sentinel].min(), "->", redate[~is_sentinel].max())


In [ ]:
# confirm labellable window

print("labellable ReDate range:",
      redate[labellable].min(), "->", redate[labellable].max()) #uses the mask tokeep only the labellable dates, then prints their span

### Cohort definition

`Attempt 1 of 3 — initial labelling approach, later superseded. Kept to show the reasoning trail.`

**Working cohort: 18,711 of 30,005 RX records.**

Excluded from the labellable set:

- **18** - placeholder-null recall dates (VisionPlus `1900-01-01` sentinel value)
- **11,276** - recall windows closing after the data-end date (Jun 2026); outcome not yet observable

Remaining **18,711** records have a real recall due date whose 6-month
grace window closes within the appointment data (latest labellable
`ReDate`: 2025-12-19), so churn outcome is observable for every row.

Patients appear in **rx** more than once on average (around 1.74 rows each), so "one row per patient" isn't automatic - we have to pick which recall stands in for each patient.

**Choice**: their most recent closed recall. This keeps the train/test split clean (a patient can't be half in train, half in test) and defines churn against a single, settled recall event rather than every recall they've ever had.







In [ ]:
print("RX rows:", len(rx))  
print("distinct patients in RX:", rx["patient_key"].nunique())  


# >1 means the average patient has multiple recall records
print("avg RX rows per patient:", round(len(rx) / rx["patient_key"].nunique(), 2))  

#### **Two decisions**

**Unit of analysis:** one row per patient (17,245 patients, 1.74 recalls each on average). We label each patient's most recent *closed* recall, rather than every recall event, to keep the train/test split clean at the patient level and avoid the same patient appearing in both.

**Return definition:** a patient "returned" if they have a `Completed` appointment within 6 months of that recall's due date (`ReDate`); otherwise `churned = 1`. Event-level labelling and re-engagement signals (patients with a later pending recall) are noted as future extensions.

### Label Construction

#### Attempt 1

In [ ]:
# ATTEMPT 1 (superseded) — most-recent-closed-recall label; kept for the reasoning trail

rx_lab = rx[labellable] #apply boolean mask to keep labellable rows only

print(len(rx_lab))

In [ ]:
print(rx_lab["ReDate"].dtype)

In [ ]:
print(rx_lab["ReDate"].head())

In [ ]:
# make ReDate a real date, then confirm

rx_lab["ReDate"] = pd.to_datetime(rx_lab["ReDate"], errors="coerce")

In [ ]:
print(rx_lab["ReDate"].dtype)                        # datetime64[ns]
print(rx_lab["ReDate"].min(), "->", rx_lab["ReDate"].max())

In [ ]:
# drop duplicates to get one row per patient, keeping the most recent ReDate

labels = rx_lab.sort_values("ReDate").drop_duplicates(subset="patient_key", keep="last")

In [ ]:
len(labels)

In [ ]:
print("distinct patients in rx_lab:", rx_lab["patient_key"].nunique())
print("rows in labels:", len(labels))

In [ ]:
# get the most recent completed appointment for each patient


completed = app[ app["Status"] == "Completed" ].copy()

completed["AppointDate"] = pd.to_datetime(completed["AppointDate"], errors="coerce")

In [ ]:
print("completed rows:", len(completed))          # expect 29,517
print("AppointDate dtype:", completed["AppointDate"].dtype)   # expect datetime64
print("date range:", completed["AppointDate"].min(), "->", completed["AppointDate"].max())

In [ ]:
# join appointments onto labels

merged = labels.merge(completed, on="patient_key", how="left")

In [ ]:
print("labels rows:", len(labels))
print("merged rows:", len(merged))

In [ ]:
print([c for c in merged.columns if "Date" in c])

In [ ]:
# determine which appointments fall within the recall window

after_recall = merged["AppointDate_y"] >= merged["ReDate"]
within_6mo   = merged["AppointDate_y"] <= merged["ReDate"] + pd.DateOffset(months=6)
merged["in_window"] = after_recall & within_6mo


print(merged["in_window"].value_counts())
print(merged["in_window"].sum(), "appointments fall in a recall window")

In [ ]:
# which patients returned within the recall window

returned = merged.groupby("patient_key")["in_window"].any()

print("patients:", len(returned))          # should be 12,198 (one per pt)
print(returned.value_counts())

In [ ]:
# churn rate = fraction of patients who did NOT return within the recall window

churn_rate = (~returned).mean()          
print(f"initial churn rate: {churn_rate:.1%}")

In [ ]:
appt_start = completed["AppointDate_y"].min() if "AppointDate_y" in completed else completed["AppointDate"].min()
print("appointments start:", appt_start)
print("recall dates before appointments start:")
print((labels["ReDate"] < appt_start).sum(), "of", len(labels))

In [ ]:
# take 5 patients marked as churned (returned == False)
churned_keys = returned[~returned].index[:5]

for k in churned_keys:
    r = labels.loc[labels["patient_key"] == k, "ReDate"].iloc[0]
    appts = completed.loc[completed["patient_key"] == k, "AppointDate"].sort_values()
    print(f"\npatient {k[:8]}  recall due: {r.date()}")
    print(f"  window: {r.date()} -> {(r + pd.DateOffset(months=6)).date()}")
    print(f"  their completed appts: {[str(a.date()) for a in appts]}")

In [ ]:
for k in churned_keys:
    r = labels.loc[labels["patient_key"] == k, "ReDate"].iloc[0]
    months = labels.loc[labels["patient_key"] == k, "ReaMonths"].iloc[0]
    appts = completed.loc[completed["patient_key"] == k, "AppointDate"].sort_values()
    last_appt = appts.max()
    print(f"patient {k[:8]}: ReDate={r.date()}, ReaMonths={months}, "
          f"last_appt={last_appt.date() if pd.notna(last_appt) else None}, "
          f"last_appt+interval={(last_appt + pd.DateOffset(months=int(months))).date() if pd.notna(last_appt) else None}")

In [ ]:
# Frame B, first pass: per patient, last completed appointment
last_appt = completed.groupby("patient_key")["AppointDate"].max()
data_end = completed["AppointDate"].max()   # 2026-06-19

# bring in each patient's recall interval (from labels)
interval = labels.set_index("patient_key")["ReaMonths"]

# align patients present in both
common = last_appt.index.intersection(interval.index)
gap_days = (data_end - last_appt[common]).dt.days
due_days = interval[common] * 30 + 180        # interval months (~30d) + 6mo grace

churned_B = gap_days > due_days
print("patients:", len(churned_B))
print(churned_B.value_counts())
print("churn rate:", round(churned_B.mean(), 3))

In [ ]:
appts_per_patient = completed.groupby("patient_key").size()
print(appts_per_patient.describe())
print("\ndistribution:")
print(appts_per_patient.value_counts().sort_index().head(10))

In [ ]:
established = appts_per_patient[appts_per_patient >= 2].index
print("established cohort (2+ appts):", len(established))
print("3+ appts:", (appts_per_patient >= 3).sum())

# churn rate on established cohort using the Frame B gap logic
last_appt = completed.groupby("patient_key")["AppointDate"].max()
data_end = completed["AppointDate"].max()
interval = labels.set_index("patient_key")["ReaMonths"]
common = established.intersection(interval.index)
gap_days = (data_end - last_appt[common]).dt.days
due_days = interval[common] * 30 + 180
churn_est = gap_days > due_days
print("established churn rate:", round(churn_est.mean(), 3))

### Cohort Scoping - Established Patients

The initial churn labelling produced an implausible 84.5% churn rate. Diagnosis revealed that: (1) `ReDate` is a *projected* next due date (which adds last exam to `ReaMonths`), not a past recall event; (2) 61% of patients(10,330 of 16,878)
have only one completed appointment in the 2020–2026 window, meaning there's insufficient history to distinguish churn from low-frequency attendance.

**Decision:** scope churn modelling to the *established* cohort, so patients with
2+ completed appointments (n=6,548). Churn defined as time since last visit
exceeding recall interval (`ReaMonths`) + 6-month grace, as of the data cutoff.
Resulting churn rate: 32%, which is consistent with the optical sector norms.

Single-visit patients represent an acquisition/first-return problem, out of
scope for this retention model.

#### Attempt 2

In [ ]:
# established cohort, for reference

est = labels[labels["patient_key"].isin(established)].copy()
print(len(est))



In [ ]:
print("established (2+ appts):", len(established))
print("of those, also in labels (have ReaMonths):", labels["patient_key"].isin(established).sum())
print("est rows:", len(est))

Established cohort with a usable recall interval: n=5,997 (551 of the 6,548
2+ visit patients had no labellable RX record and are excluded).

In [ ]:


last_appt = completed.groupby("patient_key")["AppointDate"].max() # each patient's most recent completed visit

est["last_appt"] = est["patient_key"].map(last_appt)

print(est["last_appt"].notna().sum(), "of", len(est), "have a last_appt")
print(est["last_appt"].min(), "->", est["last_appt"].max())

In [ ]:
#set data end to last appointment

data_end = completed["AppointDate"].max()  

gap_days = (data_end - est["last_appt"]).dt.days 
allowed_days = est["ReaMonths"] * 30 + 180 # churn grace period (6mo)
est["churned"] = (gap_days > allowed_days).astype(int) # 1 if churned, 0 otherwise

In [ ]:
print(est["churned"].value_counts())
print("churn rate:", round(est["churned"].mean(), 3))

In [ ]:
print("Final label — established cohort")
print("patients:", len(est))
print("churn rate:", round(est["churned"].mean(), 3))
print(est[["patient_key", "ReaMonths", "last_appt", "churned"]].head())

In [ ]:
est.to_csv(WORKING / "labels_established.csv", index=False)
print("saved:", WORKING / "labels_established.csv")

### Feature Engineering

In [ ]:
# bring each patient's last_appt onto their appointment rows
last_appt_map = est.set_index("patient_key")["last_appt"]
appts = completed[completed["patient_key"].isin(est["patient_key"])].copy()
appts["last_appt"] = appts["patient_key"].map(last_appt_map)

In [ ]:
# keep appointments at or before the patient's prediction point
appts_upto = appts[appts["AppointDate"] <= appts["last_appt"]]

In [ ]:
visit_count = appts_upto.groupby("patient_key").size()

In [ ]:
# attach to est
est["visit_count"] = est["patient_key"].map(visit_count)

In [ ]:
print(est["visit_count"].describe())
print("any missing?", est["visit_count"].isna().sum())
print("minimum:", est["visit_count"].min())

In [ ]:
#tenure days

first_appt = appts_upto.groupby("patient_key")["AppointDate"].min()  
est["first_appt"] = est["patient_key"].map(first_appt)
est["tenure_days"] = (est["last_appt"] - est["first_appt"]).dt.days     #how long have they been a patient?

In [ ]:
print(est["tenure_days"].describe())
print("any negative?", (est["tenure_days"] < 0).sum())
print("any missing?", est["tenure_days"].isna().sum())

In [ ]:
# average gap days

est["avg_gap_days"] = est["tenure_days"] / (est["visit_count"] - 1)


print(est["avg_gap_days"].describe())
print("any infinite?", np.isinf(est["avg_gap_days"]).sum())   # needs: import numpy as np
print("any missing?", est["avg_gap_days"].isna().sum())

In [ ]:
# inspect order table 

print(orders.columns.tolist())
print(orders[["OrderDate", "Amount"]].head())
print("OrderDate dtype:", orders["OrderDate"].dtype)
print("Amount dtype:", orders["Amount"].dtype)

In [ ]:
# convert OrderDate to datetime, then check for unparseable values


orders["OrderDate"] = pd.to_datetime(orders["OrderDate"], errors="coerce")
print("OrderDate dtype:", orders["OrderDate"].dtype)
print("unparseable dates:", orders["OrderDate"].isna().sum())

In [ ]:
# attach each patient's last_appt onto their order rows, then keep only orders at or before the last_appt


orders_c = orders[orders["patient_key"].isin(est["patient_key"])].copy()
orders_c["last_appt"] = orders_c["patient_key"].map(last_appt_map)
orders_upto = orders_c[orders_c["OrderDate"] <= orders_c["last_appt"]]

In [ ]:
# total spend and order count


est["total_spend"] = est["patient_key"].map(orders_upto.groupby("patient_key")["Amount"].sum())
est["order_count"] = est["patient_key"].map(orders_upto.groupby("patient_key").size())

In [ ]:
#zero orders if total spend = 0

est["total_spend"] = est["total_spend"].fillna(0)
est["order_count"] = est["order_count"].fillna(0)

In [ ]:
print(est[["total_spend", "order_count"]].describe())
print("patients with zero orders:", (est["order_count"] == 0).sum())

In [ ]:
# --- Demographics (straight lookup, no date filter) ---
by  = pat.set_index("patient_key")["birth_year"]
sex = pat.set_index("patient_key")["Sex"]
est["birth_year"] = est["patient_key"].map(by)
est["sex"]        = est["patient_key"].map(sex)

# age as of last_appt (not today) // avoids leakage
est["age"] = est["last_appt"].dt.year - est["birth_year"]

In [ ]:
print(est["age"].describe())
print("age <0 or >110:", ((est["age"] < 0) | (est["age"] > 110)).sum())
print("sex:", est["sex"].value_counts(dropna=False).to_dict())

In [ ]:
print("oldest", est["age"].max())
print("count at 99+:", (est["age"] >= 99).sum())
print(est.loc[est["age"] >= 95, ["birth_year", "last_appt", "age"]].head())

### Label Redesign

The gap-based label ("overdue as of data-end") proved unstable: churn rate
swings 32% to 0% to 89% depending on cohort maturity slicing. Root cause: the
definition conflates genuine lapse with data-window truncation (a patient's
`last_appt` being old because that's where their history ends).


**Approach:** anchor the label to each patient's own recall clock, mirroring how
the model would score in production (rolling evaluation as new data arrives).

- **Prediction point:** each patient's last completed visit.
- **Churn:** no completed appointment within `ReaMonths + 3-month grace` of that visit.
- **Cohort:** established patients (2+ completed visits) whose full recall window
  closes on or before the data-end date (2026-06-19), so every outcome is fully
  observable. Patients whose window extends past data-end are excluded as
  unobservable (right-censored), with counts logged.


#### Attempt 3

**Step 1 — Per-patient recall clock and observable cohort.**

In [ ]:
data_end = completed["AppointDate"].max()
print("data_end:", data_end)

# each patient's last completed visit
last_appt = completed.groupby("patient_key")["AppointDate"].max()

# most recent recall interval per patient (latest RX row)
rx["ReDate"] = pd.to_datetime(rx["ReDate"], errors="coerce")
latest_rx = rx.sort_values("ReDate").drop_duplicates("patient_key", keep="last")
interval = latest_rx.set_index("patient_key")["ReaMonths"]

In [ ]:
# assemble per-patient
cohort = pd.DataFrame({"last_appt": last_appt})
cohort["ReaMonths"] = cohort.index.map(interval)

# drop patients with no recall interval (can't define a window)
print("before interval filter:", len(cohort))
cohort = cohort.dropna(subset=["ReaMonths"])
print("after interval filter:", len(cohort))

# window closes = last visit + interval months + 3 month grace
cohort["window_close"] = cohort["last_appt"] + \
    pd.to_timedelta(cohort["ReaMonths"] * 30 + 90, unit="D")

In [ ]:
# keep only patients whose window is fully observable
cohort["observable"] = cohort["window_close"] <= data_end
print("observable:", cohort["observable"].sum())
print("unobservable (right-censored, excluded):", (~cohort["observable"]).sum())

In [ ]:
visit_counts = completed.groupby("patient_key").size()
cohort["visit_count"] = cohort.index.map(visit_counts)

observable = cohort[cohort["observable"]].copy()
established_obs = observable[observable["visit_count"] >= 2]
print("observable & established (2+ visits):", len(established_obs))
print("dropped (single-visit):", len(observable) - len(established_obs))

**Step 2 — Final label.**

Anchor: each patient's second-to-last completed visit (the prediction point;
their subsequent visit, or absence of one, is the observable outcome).


Churn = no completed visit within `ReaMonths + 3mo grace` of the anchor.


Cohort: **5,029** patients with a fully-observable window (window closes ≤ data-end.)

Churn rate: **37.8%** — stable and consistent with sector norms.

In [ ]:
appts_sorted = completed.sort_values(["patient_key", "AppointDate"])

# take second-to-last and last row per patient, keeping patient_key as index
g = appts_sorted.groupby("patient_key")
index_visit = g.nth(-2).set_index("patient_key")["AppointDate"]
last_visit  = g.nth(-1).set_index("patient_key")["AppointDate"]

print(index_visit.head())
print("length:", len(index_visit))

In [ ]:
cohort2 = pd.DataFrame({"index_visit": index_visit, "last_visit": last_visit})
cohort2["ReaMonths"] = cohort2.index.map(interval)
cohort2 = cohort2.dropna(subset=["ReaMonths", "index_visit"])

# window measured from the INDEX visit now
cohort2["window_close"] = cohort2["index_visit"] + \
    pd.to_timedelta(cohort2["ReaMonths"] * 30 + 90, unit="D")

# observable = window closes within data
cohort2["observable"] = cohort2["window_close"] <= data_end
print("cohort:", len(cohort2))
print("observable:", cohort2["observable"].sum())

In [ ]:
obs = cohort2[cohort2["observable"]].copy()

# test each patient's appointments against their window
cc = completed[completed["patient_key"].isin(obs.index)].copy()
cc["index_visit"]  = cc["patient_key"].map(obs["index_visit"])
cc["window_close"] = cc["patient_key"].map(obs["window_close"])

# a return = completed visit strictly after index, on/before window close
in_window = ((cc["AppointDate"] > cc["index_visit"]) &
             (cc["AppointDate"] <= cc["window_close"]))
returned = in_window.groupby(cc["patient_key"]).any()

obs["returned"] = obs.index.map(returned).fillna(False)
obs["churned"]  = (~obs["returned"]).astype(int)

print("cohort:", len(obs))
print(obs["churned"].value_counts())
print("churn rate:", round(obs["churned"].mean(), 3))

In [ ]:
obs.to_csv(WORKING / "cohort_labelled.csv")
print("saved:", len(obs), "patients")

In [ ]:
# the prediction point for every patient
index_map = obs["index_visit"]

# appointments strictly BEFORE or AT the index visit (the leakage wall)
appts = completed[completed["patient_key"].isin(obs.index)].copy()
appts["index_visit"] = appts["patient_key"].map(index_map)
appts_upto = appts[appts["AppointDate"] <= appts["index_visit"]]

# FEATURE 1: visit_count (visits up to and including index)
obs["visit_count"] = obs.index.map(appts_upto.groupby("patient_key").size())
print(obs["visit_count"].describe())
print("min:", obs["visit_count"].min(), "| missing:", obs["visit_count"].isna().sum())

In [ ]:
# FEATURE 2: tenure_days — first visit to index visit
first_upto = appts_upto.groupby("patient_key")["AppointDate"].min()
obs["first_visit"] = obs.index.map(first_upto)
obs["tenure_days"] = (obs["index_visit"] - obs["first_visit"]).dt.days


In [ ]:
# FEATURE 3:  avg gap = tenure / (number of gaps); gaps = visits - 1
# for 1-visit patients, visits-1 = 0 -> NaN (undefined, XGBoost handles it)


gaps = obs["visit_count"] - 1
obs["avg_gap_days"] = obs["tenure_days"] / gaps.where(gaps > 0)


In [ ]:
print(obs["avg_gap_days"].describe())
print("infinities:", np.isinf(obs["avg_gap_days"]).sum())
print("NaN (expected — single-visit patients):", obs["avg_gap_days"].isna().sum())

In [ ]:
# orders up to the index visit (leakage wall)
ord_c = orders[orders["patient_key"].isin(obs.index)].copy()
ord_c["index_visit"] = ord_c["patient_key"].map(index_map)
ord_upto = ord_c[ord_c["OrderDate"] <= ord_c["index_visit"]]

# FEATURES 4 & 5
obs["total_spend"] = obs.index.map(ord_upto.groupby("patient_key")["Amount"].sum()).fillna(0)
obs["order_count"] = obs.index.map(ord_upto.groupby("patient_key").size()).fillna(0)

In [ ]:
print(obs[["total_spend", "order_count"]].describe())
print("zero-spend patients:", (obs["total_spend"] == 0).sum())

In [ ]:
by  = pat.set_index("patient_key")["birth_year"]
sex = pat.set_index("patient_key")["Sex"]
obs["birth_year"] = obs.index.map(by)
obs["sex"]        = obs.index.map(sex)

# age as of the INDEX visit (prediction point), not today
obs["age"] = obs["index_visit"].dt.year - obs["birth_year"]

print(obs["age"].describe())
print("age <0 or >110:", ((obs["age"] < 0) | (obs["age"] > 110)).sum())
print("age missing:", obs["age"].isna().sum())
print("sex:", obs["sex"].value_counts(dropna=False).to_dict())

### Modelling

Baseline XGBoost churn classifier on the temporal split (train: pre-Dec-2023
index visits, test: post). Metric: AUC (threshold-independent, robust to the
train/test base-rate shift). `avg_gap_days` NaNs handled natively by XGBoost.

In [ ]:
#assembling model table
feature_cols = ["visit_count", "tenure_days", "avg_gap_days",
                "total_spend", "order_count", "age", "sex", "ReaMonths"]

model_df = obs[feature_cols + ["churned", "index_visit"]].copy()

# encode sex BEFORE splitting
model_df["sex"] = (model_df["sex"] == "M").astype(int)

# temporal split on index_visit (the prediction point)
model_df = model_df.sort_values("index_visit")
cutoff = model_df["index_visit"].quantile(0.8)
train = model_df[model_df["index_visit"] < cutoff].copy()
test  = model_df[model_df["index_visit"] >= cutoff].copy()

print("cutoff:", cutoff)
print("train:", len(train), "| test:", len(test))
print("train churn:", round(train["churned"].mean(), 3))
print("test churn:", round(test["churned"].mean(), 3))

**both non-zero, both plausible**

Train *40.3%*, test *27.7%*

- Mild distribution shift
  






In [ ]:
#model 

train.to_csv(WORKING / "train.csv")
test.to_csv(WORKING / "test.csv")
print("saved train:", len(train), "test:", len(test))